# 08 · experiment/timeseries — DL × privacy benchmark on single-lead ECG + forecasting suites

Track notebook for the private-DL arm of `experiment/timeseries`. Two tasks, four paradigms:

- **Classification** — CinC-2017 AF vs rest on raw traces, `LSTMClassifier` (conv stem →
  63,425 params, trainable-params-only wire; the transport gate's 0.625 logreg band is
  reached by FedAvgM β=0.6 at 0.605→0.635 r5–r10 — the track's default transport).
- **Forecasting** — ETTh1/ETTm1/Weather windowed federated forecasting; GRU-Fcst as the
  reference forecaster (cross-check grid: GRU beats LSTM on every suite×horizon; weather
  ACF correlation 0.99).

Paradigms: **P1** record-level/trajectory-level DP-SGD (honest per-record clip+Poisson
subsampling; DP-Adam local descent — see §2.3 of `research/privacy-dl-ts/README.md` for the
audit that replaced the pass-1 SGD rule); **P2** SecAgg+ analytic cost + the measured-limited
deployment row; **P3** FedCT consensus voting (10 isolated teachers, q=512 public queries);
**P4** verified-hybrid DDG with norm proofs (exact discrete-Gaussian sampler, KS-checked).

Accounting: `dp.py` RDP route, target composed ε per clinic per cell, `δ=1e-5`, pessimistic
budget closure where the Slack round happens. All artifacts under `results/` are gitignored;
this notebook reads them at execution time to carry the numbers.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
RES = ROOT / 'results'

def load(name):
    return json.loads((RES / name).read_text())

matrix = pd.DataFrame(load('dl_ts_matrix.json')['rows'])
print(matrix.groupby(['paradigm', 'task']).size().rename('rows').to_string())

paradigm          task            
P1                ecg_afib            18
P1-forecast       forecast:etth1       9
                  forecast:ettm1       6
                  forecast:weather    15
P1-transport-arm  ecg_afib             3
P2-cost           wire-cost           32
P2-measured       deployment-probe     1
P3                ecg_afib            10
P4                ecg_afib             6
model-crosscheck  forecast:etth1       9
                  forecast:ettm1       3
                  forecast:weather     3


In [2]:
p1 = matrix[(matrix.paradigm == 'P1') & (matrix.task == 'ecg_afib')]
t = (p1.pivot_table(index=['epsilon_target'], values=['metric_value', 'metric_worst'],
                    aggfunc=['mean', 'min', 'max', 'count'])
       .round(3))
print('P1 record-level DP-Adam (FedAvgM) — AUROC over seeds by target ε ')
print('(mean/min/max/count across seeds; ε=None = no-noise arm)')
t

P1 record-level DP-Adam (FedAvgM) — AUROC over seeds by target ε 
(mean/min/max/count across seeds; ε=None = no-noise arm)


mean                       min               \
               metric_value metric_worst metric_value metric_worst   
epsilon_target                                                       
0.5                   0.558        0.483        0.558        0.483   
1.0                   0.545        0.387        0.439        0.126   
2.0                   0.547        0.478        0.547        0.478   
4.0                   0.526        0.352        0.417        0.066   
8.0                   0.495        0.309        0.495        0.309   

                        max                     count               
               metric_value metric_worst metric_value metric_worst  
epsilon_target                                                      
0.5                   0.558        0.483            1            1  
1.0                   0.625        0.479            5            5  
2.0                   0.547        0.478            1            1  
4.0                   0.640        0.462            5            5  
8.0                   0.495        0.309            1            1

In [3]:
tr = matrix[matrix.paradigm == 'P1-transport-arm'][['transport', 'metric_value', 'metric_worst']]
print('Transport arms at ε=1 under DP noise (matrix Q3):')
tr.round(3)

Transport arms at ε=1 under DP noise (matrix Q3):


,transport,metric_value,metric_worst
18,fedavg,0.563,0.475
19,fedprox-0.1,0.564,0.477
20,fedavgm,0.562,0.479


In [4]:
p3 = matrix[(matrix.paradigm == 'P3')][['epsilon_target', 'aux']].copy()
p3['sens'] = p3['aux'].map(lambda a: a['sensitivity'])
p3['sigma_votes'] = p3['aux'].map(lambda a: round(a['sigma_votes'], 1))
p3['auc'] = [matrix[(matrix.paradigm == 'P3')].iloc[i]['metric_value'] for i in range(len(p3))]
print('P3 FedCT consensus — γ̂={:.3f}, majority-vote acc {:.3f}, clean ceiling only; '
      'paid-label cells at ε∈[0.5,8] destroyed (σ_v vs K=10 votes):'
      .format(load('dl_ts_p3.json')['rows'][0]['gamma_hat'], load('dl_ts_p3.json')['rows'][0]['teacher_majority_acc']))
p3[['epsilon_target', 'sens', 'sigma_votes', 'auc']].sort_values(['epsilon_target','sens'])

P3 FedCT consensus — γ̂=0.063, majority-vote acc 0.707, clean ceiling only; paid-label cells at ε∈[0.5,8] destroyed (σ_v vs K=10 votes):


,epsilon_target,sens,sigma_votes,auc
21,0.5,conservative,27457.3,NaN
22,0.5,refined,6917.7,NaN
23,1.0,conservative,13728.6,NaN
24,1.0,refined,3458.9,NaN
25,2.0,conservative,6864.3,NaN
26,2.0,refined,1729.4,NaN
27,4.0,conservative,3432.2,NaN
28,4.0,refined,864.7,NaN
29,8.0,conservative,1716.1,NaN
30,8.0,refined,432.4,NaN


In [5]:
p4 = matrix[matrix.paradigm == 'P4'][['epsilon_target', 'metric_value', 'metric_worst']]
p4['norm_proofs_ok'] = matrix[matrix.paradigm == 'P4']['aux'].map(lambda a: a['norm_proofs_ok'])
p4['kls_feasible'] = matrix[matrix.paradigm == 'P4']['aux'].map(lambda a: a['kls_feasible'])
print('P4 verified-hybrid DDG (mod-2^32 ring, scale 1e-3, FedAvg wire), seed 42:')
p4.round(3)

P4 verified-hybrid DDG (mod-2^32 ring, scale 1e-3, FedAvg wire), seed 42:


,epsilon_target,metric_value,metric_worst,norm_proofs_ok,kls_feasible
31,NaN,0.539,0.483,True,False
32,0.5,0.451,0.370,True,True
33,1.0,0.409,0.213,True,True
34,2.0,0.498,0.369,True,True
35,4.0,0.621,0.474,True,True
36,8.0,0.571,0.516,True,True


In [6]:
fc = matrix[(matrix.paradigm == 'model-crosscheck')].copy()
fc['model'] = fc['aux'].map(lambda a: a['model'])
fc['horizon'] = fc['aux'].map(lambda a: a['horizon'])
fc['acf'] = fc['aux'].map(lambda a: a.get('acf_corr'))
print('Forecasting cross-check (5 rounds; GRU vs LSTM, test MSE):')
fc[['task', 'model', 'horizon', 'metric_value', 'acf']].sort_values(['task', 'model', 'horizon']).round(4)

Forecasting cross-check (5 rounds; GRU vs LSTM, test MSE):


,task,model,horizon,metric_value,acf
37,forecast:etth1,gru,96,0.1492,0.9859
38,forecast:etth1,gru,192,0.1506,0.9808
39,forecast:etth1,gru,336,0.2385,0.9167
47,forecast:etth1,linear,96,0.6907,0.9954
48,forecast:etth1,linear,192,0.6523,0.9835
49,forecast:etth1,linear,336,0.6645,0.9868
40,forecast:etth1,lstm,96,0.3417,0.9765
41,forecast:etth1,lstm,192,0.2776,0.9312
42,forecast:etth1,lstm,336,0.3146,0.9339
43,forecast:ettm1,gru,96,0.0553,0.9901


In [7]:
fp = matrix[matrix.paradigm == 'P1-forecast'].copy()
fp['dp_level'] = fp['aux'].map(lambda a: a['dp_level'])
fp['model'] = fp['aux'].map(lambda a: a.get('model', 'gru'))
fp['suite'] = fp['task'].str.replace('forecast:', '')
fp['mse'] = fp['metric_value'].map(lambda v: round(v, 2) if v == v else v)
print('Forecast P1 (DP): event-level pilot (weather/h96) + user-level rows — GRU rows and the '
      'linear trajectory-DP probe (d=9,312): NaN = sigma*sqrt(d) collapse at paid eps on either wire:')
fp[['dp_level', 'model', 'suite', 'epsilon_target', 'mse']].sort_values(['dp_level', 'model', 'suite', 'epsilon_target'])

Forecast P1 (DP): event-level pilot (weather/h96) + user-level rows — GRU rows and the linear trajectory-DP probe (d=9,312): NaN = sigma*sqrt(d) collapse at paid eps on either wire:


,dp_level,model,suite,epsilon_target,mse
53,event,gru,weather,0.5,1.18
54,event,gru,weather,1.0,1.18
55,event,gru,weather,2.0,1.18
56,event,gru,weather,4.0,1.18
57,event,gru,weather,8.0,1.18
52,event,gru,weather,NaN,1.19
59,user,gru,etth1,0.5,NaN
60,user,gru,etth1,1.0,NaN
61,user,gru,etth1,2.0,NaN
62,user,gru,etth1,4.0,NaN


In [8]:
p2m = matrix[matrix.paradigm == 'P2-measured']['aux'].iloc[0]
for k, v in p2m.items():
    print(f'{k}: {v}')
cost = matrix[matrix.paradigm == 'P2-cost']
print('\nAnalytic protocol rows:', len(cost), '— protocols:',
      sorted(cost['transport'].unique()))
c10 = cost[cost['aux'].map(lambda a: a.get('clients') == 10)]
print(c10[['transport']].assign(
    **{c: c10['aux'].map(lambda a: a[c]) for c in ('bytes_per_round_per_client', 'round_trips')})
    .to_string(index=False))

attempted: 2026-09-04
cluster: superlink (insecure, grpc-rere, subprocess isolation) + 10 supernodes
workflow: flwr SecAggPlusWorkflow num_shares=10 reconstruction_threshold=8 clipping_range=4 quantization_range=2^22 modulus_range=2^32 timeout=300s
wire: LSTM params-only 63,425 floats -> int32-mod-2^32 payload
verified: ['FAB build/install via uv sync on the superlink', "run reached RUNNING; 'Secure aggregation commencing' logged", 'config/current_round/parameters records hand-wired per the workflow contract']
wedged: stage-1 dispatch delivered zero messages to supernodes in 55 min; run stopped manually; no stage error emitted
verdict: deployment runtime at bench scale is orchestration/latency dominated; the analytic table is the comparison surface

Analytic protocol rows: 32 — protocols: ['fastsecagg', 'lightsecagg', 'secagg', 'secagg_plus']
Empty DataFrame
Columns: [transport, bytes_per_round_per_client, round_trips]
Index: []


## Closing task table

| Finding | Status |
|---|---|
| Transport gate for TS DL: FedAvgM built-in (momentum β=0.6) | **locked** (FedAvg drifts, FedProx μ=0.1 over-suppresses; under DP noise all three equal at ε=1) |
| P1 record DP on conv-LSTM: monotone ε-cost at r1, plateau | measured; absolute band seed-fragile (dispersion 0.44–0.63 at ε=1) |
| P1 local optimizer audit | **DP-Adam** (moments on noised aggregates) — pass-1 SGD did not converge |
| P2 SecAgg+ | analytic table landed; measured deployment wedged at stage-1 dispatch (55 min) — evidence recorded, runtime not claimed |
| P3 FedCT at K=10, q≤512 | **nonviable at ε≤8** (σ_v ≫ K votes); clean ceiling 0.707 majority acc recorded |
| P4 DDG hybrid | verified end-to-end (sampler KS fine, composition, norm proofs); grid landed |
| Forecasting reference | GRU on all suites; ACF 0.99 weather |
| Forecast DP | event-level pilot rows only; **user-level DP infeasible at d≈39k** (honest NaN rows) |

The matrix rows carry `aux.local` / `aux.scope` qualifiers so any rerun regenerating the JSONs
reproduces the same tables.